In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import sys

sys.path.append('../')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             classification_report, confusion_matrix)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, Bidirectional,
                                     LSTM, Dense, Dropout, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from scipy.sparse import issparse

print("TensorFlow:", tf.__version__)
print("All imports successful!")

TensorFlow: 2.20.0
All imports successful!


In [2]:
df = pd.read_csv('../data/cleaned_data.csv')
df = df.dropna(subset=['cleaned_text'])

X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

y_train_arr = np.array(y_train)
y_test_arr  = np.array(y_test)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

Train size: 57628
Test size : 14407


In [3]:
# TF-IDF vectorizer for the statistical branch
MAX_TFIDF = 10000  # keep top 10k features for this branch

tfidf_hybrid = TfidfVectorizer(
    max_features=MAX_TFIDF,
    ngram_range=(1, 2),
    sublinear_tf=True
)

# Fit on train only
X_train_tfidf = tfidf_hybrid.fit_transform(X_train)
X_test_tfidf  = tfidf_hybrid.transform(X_test)

# Convert sparse matrix to dense for Keras
X_train_tfidf_dense = X_train_tfidf.toarray().astype('float32')
X_test_tfidf_dense  = X_test_tfidf.toarray().astype('float32')

print("TF-IDF train shape:", X_train_tfidf_dense.shape)
print("TF-IDF test shape :", X_test_tfidf_dense.shape)

# Save hybrid tfidf vectorizer
with open('../models/tfidf_hybrid.pkl', 'wb') as f:
    pickle.dump(tfidf_hybrid, f)
print("Hybrid TF-IDF vectorizer saved!")


TF-IDF train shape: (57628, 10000)
TF-IDF test shape : (14407, 10000)
Hybrid TF-IDF vectorizer saved!


In [4]:
MAX_WORDS = 50000
MAX_LEN   = 300
EMBED_DIM = 128

# Tokenizer for BiLSTM branch
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN,
                             padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN,
                             padding='post', truncating='post')

print("Sequence train shape:", X_train_pad.shape)
print("Sequence test shape :", X_test_pad.shape)

# Save tokenizer
with open('../models/hybrid_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Hybrid tokenizer saved!")

Sequence train shape: (57628, 300)
Sequence test shape : (14407, 300)
Hybrid tokenizer saved!


In [5]:
def build_hybrid_model(max_words, max_len, embed_dim, tfidf_dim):
    
    # --- Branch 1: BiLSTM (sequence input) ---
    seq_input = Input(shape=(max_len,), name='sequence_input')
    x1 = Embedding(input_dim=max_words,
                   output_dim=embed_dim,
                   input_length=max_len)(seq_input)
    x1 = Bidirectional(LSTM(128, return_sequences=False))(x1)
    x1 = Dropout(0.4)(x1)
    x1 = Dense(128, activation='relu')(x1)

    # --- Branch 2: TF-IDF (statistical input) ---
    tfidf_input = Input(shape=(tfidf_dim,), name='tfidf_input')
    x2 = Dense(256, activation='relu')(tfidf_input)
    x2 = Dropout(0.3)(x2)
    x2 = Dense(128, activation='relu')(x2)

    # --- Merge both branches ---
    merged = Concatenate()([x1, x2])
    x = Dense(128, activation='relu')(merged)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='relu')(x)
    output = Dense(1, activation='sigmoid', name='output')(x)

    model = Model(
        inputs=[seq_input, tfidf_input],
        outputs=output
    )

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


# Build model
hybrid_model = build_hybrid_model(
    max_words=MAX_WORDS,
    max_len=MAX_LEN,
    embed_dim=EMBED_DIM,
    tfidf_dim=MAX_TFIDF
)

hybrid_model.summary()

c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 300, 128)  │  6,400,000 │ sequence_input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfidf_input         │ (None, 10000)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 256)       │    263,168 │ embedding[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │  2,560,256 │ tfidf_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     32,896 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         65 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,330,433 (35.59 MB)

 Trainable params: 9,330,433 (35.59 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    '../models/hybrid_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("Training Hybrid Model...")
print("This will take 15-25 minutes on CPU\n")

history = hybrid_model.fit(
    [X_train_pad, X_train_tfidf_dense],  # two inputs
    y_train_arr,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print("\nTraining complete!")

Training Hybrid Model...
This will take 15-25 minutes on CPU

Epoch 1/10
811/811 ━━━━━━━━━━━━━━━━━━━━ 0s 907ms/step - accuracy: 0.9142 - loss: 0.1792
Epoch 1: val_accuracy improved from None to 0.97588, saving model to ../models/hybrid_best.keras

Epoch 1: finished saving model to ../models/hybrid_best.keras
811/811 ━━━━━━━━━━━━━━━━━━━━ 765s 931ms/step - accuracy: 0.9606 - loss: 0.0993 - val_accuracy: 0.9759 - val_loss: 0.0732
Epoch 2/10
811/811 ━━━━━━━━━━━━━━━━━━━━ 0s 822ms/step - accuracy: 0.9956 - loss: 0.0151
Epoch 2: val_accuracy improved from 0.97588 to 0.97640, saving model to ../models/hybrid_best.keras

Epoch 2: finished saving model to ../models/hybrid_best.keras
811/811 ━━━━━━━━━━━━━━━━━━━━ 687s 847ms/step - accuracy: 0.9953 - loss: 0.0152 - val_accuracy: 0.9764 - val_loss: 0.0834
Epoch 3/10
811/811 ━━━━━━━━━━━━━━━━━━━━ 0s 874ms/step - accuracy: 0.9980 - loss: 0.0066
Epoch 3: val_accuracy did not improve from 0.97640
811/811 ━━━━━━━━━━━━━━━━━━━━ 722s 890ms/step - accuracy: 0